## <span style ='color : cyan'> GENERATE UTILIZATION REPORT </SPAN>

In [24]:
import pandas as pd
import os
import json
import requests
from datetime import datetime, timedelta

group_name = 'Motrex All Vehicles'
ID = 26659458

# Create folders if they don't exist
os.makedirs(group_name, exist_ok=True)
os.makedirs('results', exist_ok=True)

# Define the report template
template = {
    'id': 9,
    'n': 'Test Trip Report',
    'ct': 'avl_unit_group',
    'p': '{"descr":"","bind":{"avl_unit_group":[]}}',
    'tbl': [
        {
            'n': 'unit_group_stats',
            'l': 'Statistics',
            'c': '',
            'cl': '',
            'cp': '',
            's': '["address_format","time_format","us_units","deviation"]',
            'sl': '["Address","Time Format","Measure","Deviation"]',
            'filter_order': [],
            'p': '{"address_format":"960495616_10_5","time_format":"%E.%m.%Y_%H:%M:%S","us_units":0,"deviation":"30"}',
            'sch': {'f1': 0, 'f2': 0, 't1': 0, 't2': 0, 'm': 0, 'y': 0, 'w': 0, 'fl': 0},
            'f': 0
        },
        {
            'n': 'unit_group_trips',
            'l': 'Trips',
            'c': '["time_begin","mileage"]',
            'cl': '["Beginning","Mileage"]',
            'cp': '[{},{}]',
            's': '',
            'sl': '',
            'filter_order': ['duration', 'mileage', 'base_eh_sensor', 'engine_hours', 'speed', 'stops', 'sensors', 'sensor_name', 'custom_sensors_col', 'driver', 'trailer', 'geozones_ex'],
            'p': '',
            'sch': {'f1': 0, 'f2': 0, 't1': 0, 't2': 0, 'm': 0, 'y': 0, 'w': 0, 'fl': 0},
            'f': 0
        }
    ],
    'bsfl': {'ct': 1663589579, 'mt': 1724130701}
}

def get_eid():
    p = 'accounts.json'
    with open(p) as fp:
        res = json.load(fp)
    access_token = res['track3']['access_token']
    base_url = res['track3']['base_url']
    url = f'https://hst-api.wialon.com/wialon/ajax.html?svc=token/login&params={{"token":"{access_token}"}}'
    r = requests.post(url)
    data = r.json()
    eid = data['eid']
    url2 = f'https://hst-api.wialon.com/wialon/ajax.html?svc=render/set_locale&params={{"tzOffset":134228528,"language":"en","formatDate":"%E.%m.%Y %H:%M:%S"}}&sid={eid}'
    requests.post(url2)
    return eid

def get_excel_report(ID, FROM, TO, template, eid):
    try:
        xl_url = f'https://hst-api.wialon.com/wialon/ajax.html?svc=report/export_result&params={{"format":8,"compress":0}}&sid={eid}'
        params = {
            "reportResourceId": 25960707,
            "reportTemplateId": 0,
            "reportObjectId": int(ID),
            "reportObjectSecId": 0,
            "reportTemplate": template,
            "interval": {
                "from": int(FROM.timestamp()),
                "to": int(TO.timestamp()),
                "flags": 0
            }
        }
        get_report = f'https://hst-api.wialon.com/wialon/ajax.html?svc=report/exec_report&params={json.dumps(params)}&sid={eid}'
        report = requests.post(get_report)
        if report.status_code == 200:
            r2 = requests.post(xl_url)
            return r2.content
        else:
            print(f"Error fetching report: {report.status_code}")
            return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# Get the EID
eid = get_eid()

# Define the default start date
default_start_date = datetime(2024, 10, 30)  # Default start date

# User input for custom start date and number of days
start_date_str = input(f"Enter the start date (YYYY-MM-DD) or press Enter to use default ({default_start_date.strftime('%Y-%m-%d')}): ")
num_days = int(input("Enter the number of days to generate reports for: 3"))

# Use default start date if user input is empty
if not start_date_str:
    start_date = default_start_date
else:
    try:
        start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
    except ValueError:
        print("Invalid date format. Please use YYYY-MM-DD.")
        exit()

# Generate reports for each day
for i in range(num_days):
    day_start = start_date + timedelta(days=i)
    day_end = day_start + timedelta(days=1) - timedelta(seconds=1)  # End of the day
    
    print(f"Generating report for {day_start.strftime('%Y-%m-%d')}...")

    # Ensure that each request is independent and the previous report generation is not affecting the next one
    report_data = get_excel_report(ID, day_start, day_end, template, eid)
    
    if report_data:
        date_str = day_start.strftime('%Y-%m-%d')
        filename = os.path.join(group_name, f'{date_str}.xlsx')
        with open(filename, 'wb') as file:
            file.write(report_data)
        print(f'Saved report for {date_str} as {filename}')
    else:
        print(f'No report generated for {day_start.strftime("%Y-%m-%d")}')


Generating report for 2024-10-30...
Saved report for 2024-10-30 as Motrex All Vehicles\2024-10-30.xlsx
Generating report for 2024-10-31...
Saved report for 2024-10-31 as Motrex All Vehicles\2024-10-31.xlsx


## <span style ='color : cyan'> PROCESS UTILIZATION REPORT </SPAN>

In [25]:
import openpyxl
# Path to the folder
folder_path = group_name

# Function to clean the Mileage column and modify the Beginning column
def process_excel(file_path):
    # Load the Excel file
    with pd.ExcelFile(file_path) as xls:
        # Check if 'Trips' sheet exists
        if 'Trips' in xls.sheet_names:
            df = pd.read_excel(xls, sheet_name='Trips')
            
            # Clean 'Mileage' column
            if 'Mileage' in df.columns:
                df['Mileage'] = df['Mileage'].astype(str).str.replace(' km', '').astype(float)
            
            # Modify 'Beginning' column
            if 'Beginning' in df.columns:
                # Convert to string for manipulation
                df['Beginning'] = df['Beginning'].astype(str)
                
                # Extract valid integer values
                valid_integers = df['Beginning'].apply(lambda x: x[:2] if x[:2].isdigit() else None).dropna().unique()
                
                # Replace '-----' with the first valid integer or a default value
                replacement_value = valid_integers[0] if valid_integers.size > 0 else '00'
                df['Beginning'] = df['Beginning'].replace('-----', replacement_value)
                
                # Keep only the first two digits
                df['Beginning'] = df['Beginning'].apply(lambda x: x[:2] if x[:2].isdigit() else replacement_value)
            
            # Save the updated DataFrame back to the Excel file
            with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                df.to_excel(writer, sheet_name='Trips', index=False)
        
        # Remove the 'Content' sheet if it exists
        if 'Content' in xls.sheet_names:
            # Open the workbook with openpyxl to delete the sheet
            wb = openpyxl.load_workbook(file_path)
            if 'Content' in wb.sheetnames:
                del wb['Content']
                wb.save(file_path)

# Iterate over all Excel files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.xlsx'):
        file_path = os.path.join(folder_path, filename)
        process_excel(file_path)

print("Processing complete.")



C:\Users\Paul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Paul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Paul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Paul\AppData\Local\Packages\PythonSof

Processing complete.


## <span style ='color : cyan'> TRANSFORM AND CONCANTENATE UTILIZATION REPORT </SPAN>

In [26]:
# Path to the folder
folder_path = folder_path  
group_name = group_name    

# Function to read and combine all Excel files into a single DataFrame
def combine_files_to_dataframe(folder_path):
    combined_df = pd.DataFrame()
    
    for filename in os.listdir(folder_path):
        if filename.endswith('.xlsx'):
            file_path = os.path.join(folder_path, filename)
            with pd.ExcelFile(file_path) as xls:
                if 'Trips' in xls.sheet_names:
                    df = pd.read_excel(xls, sheet_name='Trips')
                    combined_df = pd.concat([combined_df, df], ignore_index=True)
                    
    return combined_df

# Combine all files into a single DataFrame
combined_df = combine_files_to_dataframe(folder_path)

# Assuming the 'Beginning' column values are days of the month (integers)
trips2 = combined_df.copy()

# Convert 'Beginning' to a string, and format as 'day' if needed
trips2['day'] = trips2['Beginning'].astype(int)

# Add month and year columns for July
trips2['month'] = 10  # July is the 7th month
trips2['year'] = 2024  # Replace with the actual year

# Create a datetime column from day, month, and year
trips2['Date'] = pd.to_datetime(trips2[['year', 'month', 'day']])

# Determine the day of the week
trips2['Day of Week'] = trips2['Date'].dt.day_name()

# Create a formatted day column in the format "Mo-1", "Tu-2", etc.
trips2['Day Formatted'] = trips2['Date'].dt.strftime('%a').str[:2] + '-' + trips2['day'].astype(str)

# Rename columns for clarity
trips2.rename(columns={'Mileage':'Distance(KM)', 'Grouping':'Vehicle'}, inplace=True)

# Create pivot table for utilization based on the formatted day
utilization = pd.pivot_table(trips2, values='Distance(KM)', 
                             index='Vehicle', columns='Day Formatted',
                             fill_value=0.0, aggfunc='sum')

# Sort the columns to ensure they start from the 1st day of the month
sorted_columns = sorted(utilization.columns, key=lambda x: int(x.split('-')[1]))
utilization = utilization.reindex(columns=sorted_columns)

# Calculate days with and without trips
days_with_trips = (utilization > 0.0).sum(axis=1)
days_without_trips = (utilization == 0.0).sum(axis=1)

# Create daily utilization pivot table based on day names
daily_utilization = pd.pivot_table(trips2, values='Distance(KM)', 
                                   index='Vehicle', columns=trips2['Day of Week'],
                                   fill_value=0.0, aggfunc='sum')

# Separate weekdays and weekends
weekdays = daily_utilization.columns.difference(['Saturday', 'Sunday'])
weekends = daily_utilization.columns.intersection(['Saturday', 'Sunday'])

# Calculate total distance for weekdays and weekends
utilization['Weekday Distance (km)'] = daily_utilization[weekdays].sum(axis=1)
utilization['Weekend Distance (km)'] = daily_utilization[weekends].sum(axis=1)

# Calculate total distance
utilization['Total Distance (km)'] = utilization['Weekday Distance (km)'] + utilization['Weekend Distance (km)']

# Add days with and without trips
utilization['Days With Trips'] = days_with_trips
utilization['Days Without Trips'] = days_without_trips

# Sort utilization by total distance
utilization.sort_values(by='Total Distance (km)', ascending=True, inplace=True)

# Save the final utilization report to Excel
output_file = f'{group_name} Utilization.xlsx'
utilization.to_excel(output_file)

print(f"Utilization report saved to {output_file}.")


Utilization report saved to Motrex All Vehicles Utilization.xlsx.


## <span style ='color : red'> GENERATE ECO-DRIVING REPORT </SPAN>

In [85]:
group_name = group_name
ID = ID

# Create folders if they don't exist
os.makedirs(group_name, exist_ok=True)
#os.makedirs('results', exist_ok=True)

# Define the report template
scoring_template = {'id': 152,
  'n': 'Mawa Scoring Fleet Report',
  'ct': 'avl_unit_group',
  'p': '{"descr":"","bind":{"avl_unit_group":[]}}',
  'tbl': [{'n': 'unit_group_stats',
    'l': 'Statistics',
    'c': '',
    'cl': '',
    'cp': '',
    's': '["address_format","time_format","us_units","deviation"]',
    'sl': '["Address","Time Format","Measure","Deviation"]',
    'filter_order': [],
    'p': '{"address_format":"1255211008_10_5","time_format":"%Y-%m-%E_%H:%M:%S","us_units":0,"deviation":"30"}',
    'sch': {'f1': 0,
     'f2': 0,
     't1': 0,
     't2': 0,
     'm': 0,
     'y': 0,
     'w': 0,
     'fl': 0},
    'f': 0},
   {'n': 'unit_group_ecodriving',
    'l': 'Eco driving',
    'c': '["mileage","violation_name","violations_count","violation_duration","violation_mileage"]',
    'cl': '["Mileage","Violation","Count","Violation duration","Violation mileage"]',
    'cp': '[{},{},{},{},{}]',
    's': '',
    'sl': '',
    'filter_order': ['violation_group_name',
     'violation_duration',
     'show_all_trips',
     'mileage',
     'colors',
     'custom_sensors_col',
     'geozones_ex'],
    'p': '{"violation_group_name":"*"}',
    'sch': {'f1': 0,
     'f2': 0,
     't1': 0,
     't2': 0,
     'm': 0,
     'y': 0,
     'w': 0,
     'fl': 0},
    'f': 256}],
  'bsfl': {'ct': 1724150506, 'mt': 1724150636}}


def get_eid():
    p = 'accounts.json'
    with open(p) as fp:
        res = json.load(fp)
    access_token = res['track3']['access_token']
    base_url = res['track3']['base_url']
    url = f'https://hst-api.wialon.com/wialon/ajax.html?svc=token/login&params={{"token":"{access_token}"}}'
    r = requests.post(url)
    data = r.json()
    eid = data['eid']
    url2 = f'https://hst-api.wialon.com/wialon/ajax.html?svc=render/set_locale&params={{"tzOffset":134228528,"language":"en","formatDate":"%E.%m.%Y %H:%M:%S"}}&sid={eid}'
    requests.post(url2)
    return eid

def get_excel_report(ID, FROM, TO, scoring_template, eid):
    try:
        xl_url = f'https://hst-api.wialon.com/wialon/ajax.html?svc=report/export_result&params={{"format":8,"compress":0}}&sid={eid}'
        params = {
            "reportResourceId": 17082202,
            "reportTemplateId": 0,
            "reportObjectId": int(ID),
            "reportObjectSecId": 0,
            "reportTemplate": scoring_template,
            "interval": {
                "from": int(FROM.timestamp()),
                "to": int(TO.timestamp()),
                "flags": 0
            }
        }
        get_report = f'https://hst-api.wialon.com/wialon/ajax.html?svc=report/exec_report&params={json.dumps(params)}&sid={eid}'
        report = requests.post(get_report)
        if report.status_code == 200:
            r2 = requests.post(xl_url)
            return r2.content
        else:
            print(f"Error fetching report: {report.status_code}")
            return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# Get the EID
eid = get_eid()

# Define the default start date
default_start_date = datetime(2024, 7, 28)  # Default start date

# User input for custom start date and number of days
start_date_str = input(f"Enter the start date (YYYY-MM-DD) or press Enter to use default ({default_start_date.strftime('%Y-%m-%d')}): ")
num_days = int(input("Enter the number of days to generate reports for: 3"))

# Use default start date if user input is empty
if not start_date_str:
    start_date = default_start_date
else:
    try:
        start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
    except ValueError:
        print("Invalid date format. Please use YYYY-MM-DD.")
        exit()

# Generate reports for each day
for i in range(num_days):
    day_start = start_date + timedelta(days=i)
    day_end = day_start + timedelta(days=1) - timedelta(seconds=1)  # End of the day
    
    print(f"Generating report for {day_start.strftime('%Y-%m-%d')}...")

    # Ensure that each request is independent and the previous report generation is not affecting the next one
    report_data = get_excel_report(ID, day_start, day_end, scoring_template, eid)
    
    if report_data:
        date_str = day_start.strftime('%Y-%m-%d')
        filename = os.path.join(group_name, f'{date_str}.xlsx')
        with open(filename, 'wb') as file:
            file.write(report_data)
        print(f'Saved report for {date_str} as {filename}')
    else:
        print(f'No report generated for {day_start.strftime("%Y-%m-%d")}')


Generating report for 2024-07-28...
Saved report for 2024-07-28 as Motrex All Vehicles\2024-07-28.xlsx


## <span style ='color : red'> PROCESS ECO-DRIVING REPORT </SPAN>

In [86]:
# Path to the folder containing the Excel files
folder_path = 'Motrex All Vehicles'

# Path to the Scoring folder where the pivot files will be saved
scoring_folder = 'Scoring'

# Create the Scoring folder if it doesn't exist
if not os.path.exists(scoring_folder):
    os.makedirs(scoring_folder)

# Get a list of all files in the folder
file_list = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]

# Process each file
for file_name in file_list:
    file_path = os.path.join(folder_path, file_name)
    
    # Load the Excel file into a DataFrame
    df = pd.read_excel(file_path, sheet_name='Eco driving')
    
    # Filter out rows where 'Violation' column is '-----'
    df_filtered = df[df['Violation'] != '-----']
    
    # Create the pivot table
    pivot_table = pd.pivot_table(
        df_filtered,
        index='Grouping',  # Replace with your actual grouping column name
        columns='Violation',
        values='Count',
        aggfunc='sum',  # Use sum to aggregate the counts
        fill_value=0  # Fill NaN values with 0
    )
    
    # Save the pivot table to a new Excel file in the Scoring folder with the same name
    pivot_table_file_path = os.path.join(scoring_folder, f'pivot_{file_name}')
    pivot_table.to_excel(pivot_table_file_path)

    print(f"Pivot table created and saved as '{pivot_table_file_path}'")


C:\Users\Paul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Pivot table created and saved as 'Scoring\pivot_2024-07-01.xlsx'


C:\Users\Paul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Pivot table created and saved as 'Scoring\pivot_2024-07-02.xlsx'


C:\Users\Paul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


...

Pivot table created and saved as 'Scoring\pivot_2024-07-29.xlsx'


C:\Users\Paul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Pivot table created and saved as 'Scoring\pivot_2024-07-30.xlsx'


C:\Users\Paul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Pivot table created and saved as 'Scoring\pivot_2024-07-31.xlsx'


## <span style ='color : red'> CONCATENATE ECO-DRIVING REPORT </SPAN>

In [87]:
# Define the directory containing the Excel files
directory = 'Scoring'

# List all files in the directory that start with 'pivot' and end with '.xlsx'
files = [f for f in os.listdir(directory) if f.startswith('pivot') and f.endswith('.xlsx')]

# Initialize an empty DataFrame to hold all data
combined_df = pd.DataFrame()

# Loop through each file and concatenate the data
for file in files:
    file_path = os.path.join(directory, file)
    df = pd.read_excel(file_path, sheet_name='Sheet1')
    combined_df = pd.concat([combined_df, df], ignore_index=True)

# Check for columns that need to be summed and ensure 'Grouping' column is present
if 'Grouping' not in combined_df.columns:
    raise ValueError("'Grouping' column not found in the data")

# List all numeric columns that need to be summed (assuming they are violation counts)
# You may need to adjust this list based on your actual data
violation_columns = [col for col in combined_df.columns if pd.api.types.is_numeric_dtype(combined_df[col]) and col != 'Grouping']

# Group by 'Grouping' and sum up the violation counts
aggregated_df = combined_df.groupby('Grouping')[violation_columns].sum().reset_index()

# Save the aggregated DataFrame to a new Excel file
output_file = os.path.join(directory, 'scoring_violations.xlsx')
aggregated_df.to_excel(output_file, index=False)

print("Aggregation completed and saved to", output_file)


Aggregation completed and saved to Scoring\scoring_violations.xlsx
